In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
import joblib
import os
from pathlib import Path

In [ ]:
# Detectar de manera automática la raíz del proyecto basándose en la posición de este archivo
# (Si estás usando un Jupyter Notebook .ipynb en lugar de un script .py, cambia esta línea por: raiz_proyecto = Path.cwd())
carpeta_notebooks = Path.cwd()

raiz_proyecto = carpeta_notebooks.parent

# Definir la ruta de la carpeta 'models' en la raíz y crearla si no existe
carpeta_models = raiz_proyecto / "models"
carpeta_models.mkdir(parents=True, exist_ok=True)

#print(f"📁 Raíz del proyecto detectada en: {raiz_proyecto}")
#print(f"💾 Los modelos se guardarán limpiamente en: {carpeta_models}\n")

📁 Raíz del proyecto detectada en: c:\Users\Flor\Desktop\Programacion\DA-Project-Regression-Grupo2
💾 Los modelos se guardarán limpiamente en: c:\Users\Flor\Desktop\Programacion\DA-Project-Regression-Grupo2\models



In [41]:
# Construimos la ruta absoluta usando la raíz para que no falle nunca
csv_path = raiz_proyecto / "data" / "clean" / "players_clean.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo en la ruta absoluta: {csv_path}")

df = pd.read_csv(csv_path)

print("⚙️ Procesando características matemáticas para el modelo...")

⚙️ Procesando características matemáticas para el modelo...


In [42]:
# Asegurar que no haya nulos en la edad (usamos tu columna 'age' directa)
df['age'] = df['age'].fillna(df['age'].median())

In [43]:
# Rellenar alturas vacías o en 0 con la mediana
df['height_in_cm'] = df['height_in_cm'].fillna(df['height_in_cm'].median())

In [44]:
# Mapear las posiciones categóricas (texto) a números discretos
position_map = {'Attack': 1, 'Midfield': 2, 'Defender': 3, 'Goalkeeper': 4, 'Missing': 0}
df['position_encoded'] = df['position'].map(position_map).fillna(0)

In [46]:
# Asegurar que la columna del valor más alto no contenga nulos
df['market_value_in_eur'] = df['market_value_in_eur'].fillna(0)

In [47]:
#SELECCIÓN DE VARIABLES (FEATURES Y TARGET)

# Definimos las columnas de entrada (X) y la columna que queremos predecir (y)
features = ['age', 'height_in_cm', 'market_value_in_eur', 'position_encoded']
X = df[features]
y = df['market_value_in_eur'].fillna(0)

In [48]:
# Dividir el set de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [49]:
# ENTRENAR Y EXPORTAR LOS 3 MODELOS DE ENSEMBLE
modelos = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
}

print("\n🚀 Iniciando entrenamiento de los modelos de Ensemble...")


🚀 Iniciando entrenamiento de los modelos de Ensemble...


In [ ]:
for nombre, modelo in modelos.items():
    print(f"⏱️ Entrenando {nombre}...")
    modelo.fit(X_train, y_train)
    
    # Evaluación de precisión R²
    score_train = modelo.score(X_train, y_train)
    score_test = modelo.score(X_test, y_test)
    print(f"   📊 {nombre} -> Train R²: {score_train:.4f} | Test R²: {score_test:.4f}")
    
    # Construcción del nombre físico guardando en la raíz del proyecto
    filename = carpeta_models / f"{nombre.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(modelo, filename)
    #print(f"   💾 Guardado correctamente en: '{filename}'\n")

print("✨ ¡Todos los modelos de Ensemble han sido entrenados y exportados a la raíz con éxito!")

⏱️ Entrenando Random Forest...
   📊 Random Forest -> Train R²: 0.9997 | Test R²: 0.9994
   💾 Guardado correctamente en: 'c:\Users\Flor\Desktop\Programacion\DA-Project-Regression-Grupo2\models\random_forest_model.pkl'

⏱️ Entrenando Gradient Boosting...
   📊 Gradient Boosting -> Train R²: 1.0000 | Test R²: 1.0000
   💾 Guardado correctamente en: 'c:\Users\Flor\Desktop\Programacion\DA-Project-Regression-Grupo2\models\gradient_boosting_model.pkl'

⏱️ Entrenando XGBoost...
   📊 XGBoost -> Train R²: 1.0000 | Test R²: 0.9989
   💾 Guardado correctamente en: 'c:\Users\Flor\Desktop\Programacion\DA-Project-Regression-Grupo2\models\xgboost_model.pkl'

✨ ¡Todos los modelos de Ensemble han sido entrenados y exportados a la raíz con éxito!
